# Pipeline Massiva A100: 3 Estágios de Verificação e Pesquisa HRSBF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/contatofilipevono-rgb/hrsbf-bent-verification/blob/colab-a100-2026-10-06/RUN_A100_PIPELINE.ipynb)

Este notebook executa os **3 estágios de alta demanda computacional** na GPU NVIDIA A100 em sequência:

| Estágio | Alvo | Espaço de Busca | Carga / Tempo Estimado |
|---|---|---|---|
| **Estágio 1** | **Quárticas Homogêneas em $n=10$** | $2^{22} = 4.194.304$ funções | ~15 a 30 segundos |
| **Estágio 2** | **Exaustivo $n=14$ (Grau $\le 3$)** | $2^{33} = 8.589.934.592$ funções | ~1h a 2h na A100 |
| **Estágio 3** | **Campanha Quíntica em $n=16$ (Grau 5)** | Amostragem adaptativa em $2^{273}$ | ~1 hora (configurável) |


## 1. Setup do Ambiente e Detecção da A100

In [ ]:
import os, subprocess, pathlib

REPO = 'https://github.com/contatofilipevono-rgb/hrsbf-bent-verification.git'
BRANCH = 'colab-a100-2026-10-06'
ROOT = pathlib.Path('/content/hrsbf-bent-verification')

if not (ROOT / '.git').exists():
    print('Clonando repositório na branch colab-a100-2026-10-06...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, str(ROOT)], check=True)
else:
    print('Atualizando repositório...')
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'pull', 'origin', BRANCH], check=True)

os.chdir(ROOT)
print(f'Diretório de trabalho: {os.getcwd()}')

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python pesquisa_t32/gpu_probe.py

## 2. Executar Toda a Pipeline em Sequência (Estágios 1, 2 e 3)
Executa os três estágios de ponta a ponta com checkpointing automático.

In [ ]:
# Executa os 3 estágios em sequência
# n14-chunk-log2=13 usa batches de 8192 na GPU A100 para máxima taxa de transferência
# stage3-seconds=3600 roda 1 hora de busca quíntica adaptativa
!python run_a100_pipeline.py --stages 1,2,3 --n14-chunk-log2 13 --stage3-seconds 3600

## 3. Visualizar Relatórios dos 3 Estágios

In [ ]:
import json, pathlib

print('='*70)
print('RELATÓRIO ESTÁGIO 1: Quárticas em n=10')
p1 = pathlib.Path('results/stage1_quartic_n10/result_quartic_n10.json')
if p1.exists():
    r1 = json.loads(p1.read_text())
    print(f"Testados: {r1['tested']} | Quárticas bent: {r1['homogeneous_quartic_bent']} | Status: {r1['status']} | Tempo: {r1['elapsed_seconds']:.1f}s")
else:
    print('Estágio 1 ainda não executado.')

print('='*70)
print('RELATÓRIO ESTÁGIO 2: Exaustivo n=14 (Grau <= 3)')
p2 = pathlib.Path('results/stage2_rs_bent_n14/result_n14.json')
if p2.exists():
    r2 = json.loads(p2.read_text())
    print(f"Testados: {r2['tested']}/{r2['expected']} | Bent: {r2['bent_affine_representatives']} | Bent sem P_14: {r2['bent_without_Pn']} | Cúbicas Homog: {r2['homogeneous_cubic_bent']}")
    print(f"Tempo: {r2['elapsed_seconds']/60:.2f} min ({r2['elapsed_seconds']:.1f}s)")
else:
    print('Estágio 2 ainda em andamento ou não executado.')

print('='*70)
print('RELATÓRIO ESTÁGIO 3: Campanha Quíntica em n=16')
p3 = pathlib.Path('results/stage3_quintic_n16/summary.json')
if p3.exists():
    r3 = json.loads(p3.read_text())
    print(f"Sorteios avaliados: {r3['tested_total']} | Rounds: {r3['rounds_completed']} | Fibras cobertas: {len(r3.get('global_sample_cover', []))} | Tempo: {r3['elapsed_seconds']/60:.2f} min")
else:
    print('Estágio 3 ainda não executado.')
print('='*70)

## 4. Download Completo de Todos os Resultados (.zip)

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive('resultados_a100_completo', 'zip', 'results')
    files.download('resultados_a100_completo.zip')
    print('Download de resultados_a100_completo.zip iniciado!')
except Exception as e:
    print(f'Resultados disponíveis localmente na pasta results/: {e}')